In [ ]:
## Important 

# B1 assumes A1 has been run at least once.

In [1]:
## Setup 

import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        candidate = current / "src"
        if candidate.exists():
            return candidate
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

SRC_DIR = find_src_dir()
sys.path.append(str(SRC_DIR))

import polars as pl
from paths import DATA_DIR, IBES_DIR

In [2]:
## Volume profile: retail activity by trading day relative to announcement

# Used to select the event window empirically rather than by convention.
# Result: flat until -2, spike at day 0/+1 (~2.5x baseline), decay by +5.

from analysis.event_window_profile import build_event_profile

profile = build_event_profile(window=30)
with pl.Config(tbl_rows=-1):
    print(profile)

daily_retail_*.parquet already exists -- skipping rebuild. Delete the folder first if you've changed build_daily_retail_activity.py.


In [3]:
## Sample representativness check

# Confirms the matched sample isn't concentrated in the 2020-21 retail boom.

import random

events = pl.read_parquet(IBES_DIR / "dispersion_events.parquet")
random.seed(42)
n_sample = 1000
if events.height > n_sample:
    idx = random.sample(range(events.height), n_sample)
    events_sample = events[idx]

year_counts = (
    events_sample.with_columns(pl.col("ANNDATS_ACT").dt.year().alias("year"))
    .group_by("year").len().sort("year")
)
with pl.Config(tbl_rows=-1):
    print(year_counts)

Matched 97,270 of 163,010 events to CBOE trading data
shape: (61, 3)
┌─────────┬─────────────────┬──────────┐
│ rel_day ┆ mean_retail_vol ┆ n_events │
│ ---     ┆ ---             ┆ ---      │
│ i64     ┆ f64             ┆ u32      │
╞═════════╪═════════════════╪══════════╡
│ -30     ┆ 705.993012      ┆ 94024    │
│ -29     ┆ 733.257719      ┆ 94122    │
│ -28     ┆ 730.185958      ┆ 94220    │
│ -27     ┆ 732.231435      ┆ 94342    │
│ -26     ┆ 722.294777      ┆ 94468    │
│ -25     ┆ 706.142591      ┆ 94606    │
│ -24     ┆ 720.85865       ┆ 94715    │
│ -23     ┆ 718.836406      ┆ 94820    │
│ -22     ┆ 706.241305      ┆ 94967    │
│ -21     ┆ 699.715598      ┆ 95126    │
│ -20     ┆ 681.803054      ┆ 95280    │
│ -19     ┆ 688.383603      ┆ 95385    │
│ -18     ┆ 694.000325      ┆ 95489    │
│ -17     ┆ 690.638102      ┆ 95654    │
│ -16     ┆ 680.298978      ┆ 95803    │
│ -15     ┆ 663.394114      ┆ 95962    │
│ -14     ┆ 681.641999      ┆ 96092    │
│ -13     ┆ 701.835271      ┆

In [ ]:
## Composition comparison: retail vs. procust, near event vs. baseline

from analysis.event_window_profile import build_composition_comparison

comparison = build_composition_comparison()
comparison

In [ ]:
## Headline results: difference in differences across all outcomes

# treat:post is the answer to "does retail shift differently from professional customers", not merely "does retail shift".

from analysis.event_window_profile import build_diff_in_diff_panel, run_diff_in_diff

outcomes = ["lt_100", "call", "otm", "otm_call", "otm_put", "itm", "open"]

results = []
for oc in outcomes:
    panel = build_diff_in_diff_panel(outcome=oc)
    m_ev = run_diff_in_diff(panel, cluster_by="event")
    m_tk = run_diff_in_diff(panel, cluster_by="ticker")
    results.append({
        "outcome": oc,
        "did_coef": m_ev.params["treat:post"],
        "p_event_clustered": m_ev.pvalues["treat:post"],
        "p_ticker_clustered": m_tk.pvalues["treat:post"],
        "n_obs": int(m_ev.nobs),
    })

results_df = pl.DataFrame(results)
with pl.Config(tbl_rows=-1):
    print(results_df)

In [ ]:
## Levels behind the DiD coefficients

# A coefficient alone doesn't show the underlying shares. Note the persistent baseline gap: retail sits at ~59.4% OTM vs procust ~54.1% even outside event windows.

for oc in ["otm", "otm_put", "lt_100"]:
    panel = build_diff_in_diff_panel(outcome=oc)
    levels = (
        panel.group_by(["participant_group", "is_near_event"])
        .agg(pl.col("share").mean().alias("mean_share"), pl.len().alias("n"))
        .sort("participant_group", "is_near_event")
    )
    print(f"--- {oc} ---")
    print(levels)
    print()

In [ ]:
## Moneyness decomposition: where does the OTM effect come from?

# The combined OTM effect decomposes almost entirely into puts:
#   otm_put  = +0.0097 (p < 0.0001)
#   otm_call = +0.0010 (p = 0.60, not significant)
#   otm      = +0.0107
# This runs against the lottery-preference literature's emphasis on OTM calls,
# and corroborates the separate call-share result (-0.0046) -- two independent
# measures pointing the same direction.
for oc in ["otm", "otm_call", "otm_put"]:
    panel = build_diff_in_diff_panel(outcome=oc)
    m = run_diff_in_diff(panel, cluster_by="ticker")
    print(f"{oc:9s} treat:post = {m.params['treat:post']:+.4f}, p = {m.pvalues['treat:post']:.4f}")

In [ ]:
## Mega cap divergence: volume weighted vs. equal weighted

# Ten tickers (AAPL, AMD, AMZN, BAC, C, FB, GE, MSF, NFLX, TSLA) behave oppositely to the rest of the universe on position size, and carry enough volume to flip the sign of any volume weighted aggregate. This is the empircal justifcation for the firm size control.

from analysis.event_window_profile import compare_weighting_schemes, compare_top_n_tickers

compare_weighting_schemes(outcome="lt_100")
print()
compare_top_n_tickers(outcome="lt_100", top_n=10)